### Weniger Bias, mehr Rendite? Hybrides Modell aus Large Language Models und Machine Learning zur Analyse von Earnings Calls
**Bachelorarbeit an der Universität St.Gallen (HSG)**\
[© 2026 Ruben Cardell](https://polyformproject.org/licenses/noncommercial/1.0.0)

---

# Ergebnisse – Vergleich der drei Versuche

Dieses Notebook fasst die Vorhersagen aller Modelle aus den drei Versuchen zu
einer gemeinsamen Ergebnistabelle zusammen. Berechnet werden je Modell und
Versuch die Long-Short Rendite pro Quartal und pro Jahr, die kumulierte Rendite,
die Trefferquote, die Volatilität, ein Sharpe-ähnlicher Quotient sowie RMSE und
MAE.

**Versionsbezeichnungen im Code:** `v3` = Versuch 1, `v4` = Versuch 2,
`v5` = Versuch 3.

## 1 Daten laden

In [ ]:
# Google Drive einbinden (Ausführung in Google Colab)
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Vorhersagen aller Modelle je Versuch (v3 = Versuch 1, v4 = Versuch 2,
# v5 = Versuch 3), erzeugt in den ML-Training-Notebooks
pred_v3 = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/pred_v3.csv')
pred_v4 = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/pred_v4.csv')
pred_v5 = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/pred_v5.csv')

In [ ]:
# y_test: bereinigte Zielvariable der Testdaten; y_actual: tatsächliche
# Kursentwicklungen aller Beobachtungen
y_test = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/y_test.csv', index_col=0).squeeze("columns")
y_actual = pd.read_csv('/content/drive/MyDrive/Bachelorarbeit/data/y_actual.csv', index_col=0).squeeze("columns")

## 2 Kennzahlen berechnen

In [ ]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

In [ ]:
# Angepasste Fassung der Funktion aus den ML-Training-Notebooks: Sie gibt die
# Kennzahlen stets zurück, statt sie auszugeben.

def long_short_returns(y_pred, y_test, actual_full, top_pct=0.2):
    """Berechnet die Long-Short Rendite eines Modells sowie ergänzende Fehlermasse.

    Je Quartal werden alle Unternehmen absteigend nach der vorhergesagten
    Kursentwicklung sortiert. Auf das obere Quantil wird eine Long-Position, auf
    das untere eine Short-Position simuliert; die mittleren 60% werden ignoriert.
    Die Quartalsrenditen werden anschliessend je Jahr aufgezinst und auf ein
    volles Jahr hochgerechnet.

    Rückgabe: (Quartalsrenditen, Jahresrenditen, RMSE, MAE).
    """
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    y_pred = pd.Series(y_pred, index=y_test.index)  # episode_id an die Vorhersagen anhängen
    actual = actual_full.loc[y_test.index]  # tatsächliche Renditen derselben episode_ids

    df = pd.DataFrame({'pred': y_pred, 'actual': actual})  # Vorhersage und Realität in einer Tabelle

    parts = df.index.to_series().str.split('_', expand=True)  # "AAL_2019_Q2" -> ['AAL', '2019', 'Q2']
    df['period'] = parts[1] + '_' + parts[2]  # als "2019_Q2" für die Gruppierung je Quartal

    results = {}  # eine Long-Short Rendite je Quartal
    for period, g in df.groupby('period'):  # über die Quartale iterieren
        g = g.sort_values('pred', ascending=False)  # nach Vorhersage sortieren, höchste zuerst
        n = max(1, int(len(g) * top_pct))  # Korbgrösse (oberste/unterste top_pct%), mindestens 1 Titel

        long_ret = g['actual'].iloc[:n].mean()  # durchschnittliche Rendite des Long-Korbs
        short_ret = g['actual'].iloc[-n:].mean()  # durchschnittliche Rendite des Short-Korbs

        results[period] = long_ret - short_ret  # Differenz als Long-Short Rendite des Quartals

    q_ret = pd.Series(results).sort_index()  # Quartalsrenditen in chronologischer Reihenfolge

    years = q_ret.index.str.split('_').str[0]  # Jahr aus dem Quartalslabel ("2019_Q2" -> "2019")

    def compound_and_annualize(group):
        n_quarters = len(group)  # tatsächlich vorhandene Quartale dieses Jahres
        compounded = (1 + group).prod() - 1  # vorhandene Quartale aufzinsen
        return (1 + compounded) ** (4 / n_quarters) - 1  # auf vier Quartale hochrechnen

    y_ret = q_ret.groupby(years).apply(compound_and_annualize)  # eine Jahresrendite je Jahr

    return q_ret, y_ret, rmse, mae

In [ ]:
def compute_results(pred_df, y_test, y_actual):
    """Berechnet alle Kennzahlen für jedes Modell eines Versuchs.

    Rückgabe: DataFrame mit einer Zeile je Modell und den Kennzahlen als Spalten.
    """
    results = {}
    for model, y_pred in pred_df.items():
        q_ret, y_ret, rmse, mae = long_short_returns(y_pred.values, y_test, y_actual)
        results[model] = {
            'Return/Quarter': q_ret.mean(),
            'Return/Year': y_ret.mean(),
            'Cumulative': (1 + q_ret).prod() - 1,
            'Win Rate': (q_ret > 0).mean(),
            'Std Dev': q_ret.std(),
            'Sharpe': q_ret.mean() / q_ret.std(),
            'RMSE': rmse,
            'MAE': mae,
        }

    cols = ['Return/Quarter', 'Return/Year', 'Cumulative', 'Win Rate', 'Std Dev', 'Sharpe', 'RMSE', 'MAE']
    return pd.DataFrame(results).T[cols]

In [ ]:
# Kennzahlen für alle drei Versuche berechnen
pred_versions = {'v3': pred_v3, 'v4': pred_v4, 'v5': pred_v5}

results_df = {version: compute_results(pred_df, y_test, y_actual)
              for version, pred_df in pred_versions.items()}

## 3 Ergebnistabelle aufbereiten

In [ ]:
# Versuche nebeneinander stellen (Spalten mehrstufig nach Versuch)
df = pd.concat(results_df.values(), axis=1, keys=results_df.keys())

# Zufallsprognose als weitere Zeile ergänzen: über 1'000 Durchläufe gemittelte
# Werte aus den ML-Training-Notebooks, identisch für alle drei Versuche
df.loc["-- Baseline --"] = [-0.0003, -0.0004, -0.0054, 0.4981, 0.0385, -0.0090, 0.1633, 0.1305] * 3

# Die Direktprognose des LLM ist in Versuch 3 keine eigene Spalte. Damit die
# Tabelle dennoch nach der Jahresrendite von Versuch 3 sortiert werden kann,
# wird ihr Wert vorübergehend eingesetzt und anschliessend wieder entfernt.
df.loc["-- LLM Benchmark --", ('v5', 'Return/Year')] = 0.143
df = df.sort_values(('v5', 'Return/Year'), ascending=False)

df.loc["-- LLM Benchmark --", ('v5', 'Return/Year')] = None

In [ ]:
# Für die Arbeit relevante Kennzahlen auswählen, Versuch 3 zuerst
df_yearly = df[[('v5', 'Return/Year'), ('v5', 'Sharpe'), ('v5', 'RMSE'),
                ('v4', 'Return/Year'), ('v4', 'Sharpe'), ('v4', 'RMSE'),
                ('v3', 'Return/Year'), ('v3', 'Sharpe'), ('v3', 'RMSE')]]

In [ ]:
# Zahlenformate je Kennzahl
fmt = {
    'Return/Year': '{:.1%}',
    'Sharpe': '{:.2f}',
    'RMSE': '{:.4f}'
}

format_dict = {col: fmt[col[1]] for col in df.columns if col[1] in fmt}

In [ ]:
# Trennlinien zwischen den Versuchsblöcken und nach den Zeilenbeschriftungen
top_levels = df_yearly.columns.get_level_values(0)
change_idx = [i for i in range(1, len(top_levels)) if top_levels[i] != top_levels[i-1]]

border_styles = [{'selector': f'.col{i}', 'props': [('border-left', '0.5px solid black')]}
                 for i in change_idx
                 ] + [{'selector': '.row_heading, .blank', 'props': [('border-right', '0.5px solid black')]}]

In [ ]:
# Fertige Ergebnistabelle (Tabelle 2 der Arbeit): je Versuch sind die beste
# Rendite, der beste Sharpe-Quotient und der tiefste RMSE fett hervorgehoben
(df_yearly.style
          .format(format_dict)
          .highlight_max(props='font-weight: bold;', subset=[('v5', 'Return/Year'), ('v4', 'Return/Year'), ('v3', 'Return/Year'),
                                                             ('v5', 'Sharpe'), ('v4', 'Sharpe'), ('v3', 'Sharpe')])
          .highlight_min(props='font-weight: bold;', subset=[('v5', 'RMSE'), ('v4', 'RMSE'), ('v3', 'RMSE')])
          .set_table_styles(border_styles, overwrite=False))